In [ ]:
import pandas as pd
from utilities import load_images, download_clip

# initialize the clip model
classifier = download_clip()

# sample labels
label_list = ["with a license plate", "without a license plate"]
color_list = ["white", "black", "red", "orange", "yellow", "green", "blue", "purple"]
vehicle_list = ["car", "bus", "motorcycle", "license plate"]

# directory with images
image_folder = r"../data/license_plate_detection/train/images"

# load images from image directory
image_set = load_images(directory= image_folder, num_img= 50, use_rand= True, img_obj= True)

# get pillow image objects and file names from loaded images
pil_images = [img[0] for img in image_set]
file_names = [img[1] for img in image_set]

# run results through clip classifier in batches
results = classifier(pil_images, candidate_labels=vehicle_list, batch_size=8)
    
prob_list = []

# iterate through results, assign probabilities
for idx, (predictions, file_name) in enumerate(zip(results, file_names)):
    df = pd.DataFrame(predictions).T
    df.columns = df.iloc[-1]
    df = df[:-1]
    df["fn"] = file_name
    prob_list.append(df)
    
    pct_complete = (idx + 1) / len(image_set)
    print(f"%{pct_complete*100:.2f} formatted")


In [ ]:
import pandas as pd
result = pd.concat(prob_list)

result.to_csv("C:/Users/Installer/Downloads/vehicle_labels.csv")

df = pd.read_csv("C:/Users/Installer/Downloads/vehicle_labels.csv")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
df_license_plate_only = df[df["car"] > .9].reset_index()

cols = 3
n = df_license_plate_only.shape[0]
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize = (12, rows * 3) )

axes_flat = axes.flatten()

for idx, row in df_license_plate_only.iterrows():
    axes_flat[idx].imshow(Image.open(row["fn"]))
    axes_flat[idx].set_title(f'Probability {row["license plate"]:.2f}')